## Recomendación: Funciones

Constantes, splits temporales, configuración y métricas de ranking compartidas

### Imports

In [ ]:
import json
import numpy as np

### Constantes

- `SEED` = 42: misma semilla en muestras y modelos

- `SPLITS`: años de entrenamiento y de prueba
    - **ajuste** (2023 → 2024): se eligen hiperparámetros y el peso `w` del híbrido
    - **final** (2023-2024 → 2025): se mide **una sola vez** con lo elegido en ajuste

- `KS`: cortes del ranking que se reportan

- `N_CAND`: candidatos por entidad que cada modelo entrega al híbrido

- `Q_WINSOR`: cuantil de $\log(1 + \text{monto})$ en que se recorta la señal (obras gigantes)

- `UMBRAL_HHI`: HHI desde el que una entidad es cautiva (el mismo de la Parte II)

- `MAX_ITER_ALS`: iteraciones de ALS

- `datos_dir`: datos que pasan de un notebook al siguiente

In [ ]:
SEED = 42
SPLITS = {"ajuste": {"train": [2023], "test": [2024]},
          "final":  {"train": [2023, 2024], "test": [2025]}}
KS = [5, 10]
N_CAND = 200
Q_WINSOR = 0.99
UMBRAL_HHI = 0.25
MAX_ITER_ALS = 10
datos_dir = Path.cwd() / "temp" / "datos"
datos_dir.mkdir(parents=True, exist_ok=True)

### Parámetros elegidos

Se eligen con el split **ajuste** y se guardan en `temp/datos/config.json`; los notebooks siguientes los leen de ahí

| Parámetro | Qué es | Criterio | Sección |
|---|---|---|---|
| `als.rank`, `als.reg`, `als.alpha`, `als.senal` | Factores, λ, α y señal de ALS | NDCG@10 en pares nuevos; empates por media entre semillas | `b_baseline_als` |
| `contenido.dim` | Dimensiones de la SVD | NDCG@10 en pares nuevos, búsqueda exacta | `c_contenido_ann` |
| `contenido.gamma` | Exponente del factor de actividad | NDCG@10 en pares nuevos | `c_contenido_ann` |
| `contenido.prof` | Vecinos que HNSW recupera antes de reordenar | recall@200 ≥ 95% frente al exacto | `c_contenido_ann` |
| `hibrido.w` | Peso de ALS en cada variante del híbrido | NDCG@10 en pares nuevos | `d_hibrido_evaluacion` |

### Configuración entre notebooks

`config.json` guarda lo que cada notebook decide (cuantil de winsorización, hiperparámetros, `w`…)

In [ ]:
def leer_config():
    ruta = datos_dir / "config.json"
    return json.loads(ruta.read_text()) if ruta.exists() else {}

def guardar_config(config):
    (datos_dir / "config.json").write_text(json.dumps(config, indent=1, ensure_ascii=False))

### Señal implícita

$$r_{ui} = \log(1 + n_{ui}) + \beta \cdot \frac{\min(\log(1 + m_{ui}),\, q)}{q}$$

- $n_{ui}$: adjudicaciones de la entidad $u$ al proveedor $i$; $m_{ui}$: monto en soles

- $q$: cuantil `Q_WINSOR` de $\log(1+m)$, calculado **solo con train**

- ALS usa la confianza $c_{ui} = 1 + \alpha\, r_{ui}$

In [ ]:
def senal(n, m, q, beta):
    return F.log1p(n) + F.lit(beta) * F.least(F.log1p(F.coalesce(m, F.lit(0.0))), F.lit(q)) / F.lit(q)

### ALS

`entrenar_als`: ajusta ALS implícito en Spark y baja los factores al driver como matrices NumPy (filas = índices enteros de train)

In [ ]:
from pyspark.ml.recommendation import ALS

def factores(df, n, k):
    p = df.toPandas(); Mx = np.zeros((n, k), dtype=np.float32)
    Mx[p.id.to_numpy()] = np.vstack(p.features.to_numpy()); return Mx

def entrenar_als(ratings, ctx, rank, reg, alpha, col, seed=SEED):
    t0 = time.time()
    m = ALS(rank=rank, regParam=reg, alpha=alpha, implicitPrefs=True, maxIter=MAX_ITER_ALS, seed=seed,
            userCol="entidad_idx", itemCol="proveedor_idx", ratingCol=col).fit(ratings)
    return factores(m.userFactors, ctx.n_u, rank), factores(m.itemFactors, ctx.n_i, rank), round(time.time() - t0, 1)

### Documentos de texto

`doc_procesos`: texto de cada proceso = `descripcion` + `items_desc` (solo si difiere) + un token `fam<código>` por familia UNSPSC adjudicada; lo usan `c_contenido_ann` y `e_analisis_sustitutos`

In [ ]:
def doc_procesos(procesos, anios):
    p = procesos.select("ocid", "entidad_key", F.concat_ws(" ", F.coalesce("descripcion", F.lit("")),
                        F.when(F.col("items_desc") != F.col("descripcion"), F.col("items_desc")).otherwise(F.lit(""))).alias("texto"))
    fam = (leer(final_dir, "hechos").where(F.col("anio").isin(anios) & (F.col("familia") != "SIN"))
           .groupBy("ocid").agg(F.concat_ws(" ", F.collect_set(F.concat(F.lit("fam"), "familia"))).alias("fams")))
    d = p.join(fam, "ocid", "left").toPandas()
    d["doc"] = d.texto.str.slice(0, 3000) + " " + d.fams.fillna("")
    return d

### Candidatos guardados

`leer_cand(split, modelo)`: top-`N_CAND` de un modelo, como `{régimen: DataFrame}`

In [ ]:
def leer_cand(split, modelo):
    d = pd.read_parquet(datos_dir / f"{split}_cand_{modelo}")
    return {reg: g.drop(columns="regimen").reset_index(drop=True) for reg, g in d.groupby("regimen")}

### Métricas de ranking

- `recs`: entidad → lista ordenada de proveedores recomendados; `verdad`: entidad → conjunto de proveedores adjudicados en test

- **Precision@K** = aciertos en el top-K / K

- **Recall@K** = aciertos en el top-K / relevantes

- **NDCG@K**: aciertos descontados por posición (1 / log2(pos + 1)), dividido entre el ideal

- **MAP@K**: promedio de la precisión en cada posición con acierto, dividido entre min(K, relevantes)

- Solo se evalúan entidades con al menos un relevante

In [ ]:
def metricas_entidad(rec, rel, k):
    top = rec[:k]
    hits = np.array([p in rel for p in top], dtype=float)
    n_hit = hits.sum()
    desc = 1 / np.log2(np.arange(2, len(top) + 2))
    ideal = desc[:min(len(rel), k)].sum()
    prec_pos = np.cumsum(hits) / np.arange(1, len(top) + 1)
    return {"precision": n_hit / k, "recall": n_hit / len(rel),
            "ndcg": float((hits * desc).sum() / ideal) if ideal > 0 else 0.0,
            "map": float((prec_pos * hits).sum() / min(len(rel), k))}

def evaluar(recs, verdad, k):
    filas = [metricas_entidad(recs.get(u, []), rel, k) for u, rel in verdad.items() if rel]
    return {m: float(np.mean([f[m] for f in filas])) for m in ["precision", "recall", "ndcg", "map"]} | {"entidades": len(filas)}

def por_entidad(recs, verdad, k, metrica="ndcg"):
    # valor de la métrica en cada entidad, en orden de `verdad` (para bootstrap pareado)
    return np.array([metricas_entidad(recs.get(u, []), rel, k)[metrica] for u, rel in verdad.items() if rel])

### Incertidumbre

- `ic_bootstrap`: intervalo al 95% de la media, remuestreando **entidades** con reemplazo

- Con `base` se remuestrea la **diferencia pareada** entre dos modelos en las mismas entidades: si el intervalo no contiene 0, la diferencia no es ruido

In [ ]:
def ic_bootstrap(v, base=None, n=1000, seed=SEED):
    v = np.asarray(v, dtype=float) - (0 if base is None else np.asarray(base, dtype=float))
    rng = np.random.default_rng(seed)
    medias = v[rng.integers(0, len(v), size=(n, len(v)))].mean(axis=1)
    return float(v.mean()), float(np.quantile(medias, .025)), float(np.quantile(medias, .975))

### Diversidad

- **Cobertura**: % del catálogo que aparece en algún top-K

- **% cola larga**: % de las recomendaciones que son proveedores con **un solo cliente** en train

- **% no ganadores**: % de las recomendaciones que son proveedores que ofertaron en train pero **nunca ganaron** (reenfoque de PQ3.2: el 66% de los pares nuevos de test va a ellos)

- **Popularidad media**: n.º medio de entidades clientes (en train) de lo recomendado

In [ ]:
def diversidad(recs, k, catalogo, cola_larga, no_ganadores, popularidad):
    todas = [p for r in recs.values() for p in r[:k]]
    if not todas:
        return {"cobertura": 0.0, "pct_cola_larga": 0.0, "pct_no_ganadores": 0.0, "popularidad_media": 0.0}
    return {"cobertura": len(set(todas)) / len(catalogo),
            "pct_cola_larga": float(np.mean([p in cola_larga for p in todas])),
            "pct_no_ganadores": float(np.mean([p in no_ganadores for p in todas])),
            "popularidad_media": float(np.mean([popularidad.get(p, 0) for p in todas]))}

### Ranking por lotes

- `rankear`: arma el top-`n` de cada entidad evaluable a partir de una función de puntaje sobre **todo** el catálogo del modelo

- `excluir = True` quita los proveedores que la entidad ya tenía en train (régimen **nuevos**: sustitutos)

- Lotes de 512 entidades: la matriz entidades × proveedores completa no cabe en memoria

In [ ]:
def rankear(ctx, puntaje, excluir, n=N_CAND, lote=512):
    filas = []
    for s in range(0, len(ctx.ev_idx), lote):
        b = ctx.ev_idx[s:s + lote]
        S = np.array(puntaje(b), dtype=np.float32)  # copia: el puntaje puede ser una vista de solo lectura
        if excluir:
            for j, u in enumerate(b):
                if u in ctx.vistos:
                    S[j, ctx.vistos[u]] = -np.inf
        top = np.argpartition(-S, n - 1, axis=1)[:, :n]
        sc = np.take_along_axis(S, top, 1); o = np.argsort(-sc, 1)
        top, sc = np.take_along_axis(top, o, 1), np.take_along_axis(sc, o, 1)
        for j, u in enumerate(b):
            ok = np.isfinite(sc[j])
            filas.append(pd.DataFrame({"entidad_key": ctx.ent_key[u], "proveedor_key": ctx.key_p[top[j][ok]],
                                       "score": sc[j][ok], "rank": np.arange(1, ok.sum() + 1)}))
    return pd.concat(filas, ignore_index=True)

def a_recs(cand):
    return cand.sort_values(["entidad_key", "rank"]).groupby("entidad_key").proveedor_key.apply(list).to_dict()

### Contexto de un split

`contexto(split)` reúne lo que necesitan todos los modelos: índices, vistos en train, verdad de test por régimen y conjuntos de diversidad

In [ ]:
from types import SimpleNamespace

def contexto(split):
    tr  = pd.read_parquet(datos_dir / f"{split}_train")
    te  = pd.read_parquet(datos_dir / f"{split}_test")
    cat = pd.read_parquet(datos_dir / f"{split}_catalogo")
    ev  = pd.read_parquet(datos_dir / f"{split}_evaluables")
    ks = set(ev.entidad_key)
    te = te[te.entidad_key.isin(ks)]
    return SimpleNamespace(
        split=split, train=tr, test=te, catalogo=cat, evaluables=ev,
        n_u=int(tr.entidad_idx.max()) + 1, n_i=int(tr.proveedor_idx.max()) + 1,
        key_p=tr.drop_duplicates("proveedor_idx").set_index("proveedor_idx").proveedor_key.sort_index().to_numpy(),
        key_e=tr.drop_duplicates("entidad_idx").set_index("entidad_idx").entidad_key.sort_index().to_numpy(),
        vistos=tr.groupby("entidad_idx").proveedor_idx.apply(np.array).to_dict(),
        vistos_key=tr.groupby("entidad_key").proveedor_key.apply(set).to_dict(),
        ev_idx=ev.entidad_idx.to_numpy(), ent_key=dict(zip(ev.entidad_idx, ev.entidad_key)),
        verdad={"todos": te.groupby("entidad_key").proveedor_key.apply(set).to_dict(),
                "nuevos": te[te.nuevo == 1].groupby("entidad_key").proveedor_key.apply(set).to_dict()},
        cautivas=set(ev.loc[ev.cautiva == 1, "entidad_key"]),
        ganadores=set(cat.loc[cat.gano_train == 1, "proveedor_key"]),
        cola_larga=set(cat.loc[cat.cola_larga == 1, "proveedor_key"]),
        no_ganadores=set(cat.loc[cat.gano_train == 0, "proveedor_key"]),
        popularidad=dict(zip(cat.proveedor_key, cat.popularidad)),
        catalogo_keys=set(cat.proveedor_key))

### Evaluación completa

`evaluar_modelo`: métricas de ranking y diversidad por régimen (`todos` / `nuevos`), grupo (`todas` / `cautivas`) y K

- `universo = "alcanzable"`: la verdad se restringe a proveedores que el modelo podía conocer, para separar el error del modelo del techo de *cold-start*

In [ ]:
def evaluar_modelo(ctx, modelo, cand, universos=("completo",), conocidos=None):
    filas = []
    for reg, df in cand.items():
        recs = a_recs(df)
        for uni in universos:
            for grupo, ents in [("todas", None), ("cautivas", ctx.cautivas)]:
                ver = {u: (r & conocidos if uni == "alcanzable" else r) for u, r in ctx.verdad[reg].items()
                       if ents is None or u in ents}
                for k in KS:
                    filas.append({"modelo": modelo, "regimen": reg, "universo": uni, "grupo": grupo, "k": k,
                                  **evaluar(recs, ver, k),
                                  **diversidad(recs, k, ctx.catalogo_keys, ctx.cola_larga, ctx.no_ganadores, ctx.popularidad)})
    return filas

### Normalización para el híbrido

Min-max de los scores de **cada entidad**, sobre sus candidatos: lleva ALS y contenido a la misma escala [0, 1]

In [ ]:
def minmax_por_entidad(df, col, por="entidad_key"):
    g = df.groupby(por)[col]
    mn, mx = g.transform("min"), g.transform("max")
    return ((df[col] - mn) / (mx - mn).replace(0, np.nan)).fillna(1.0)